# 04b - Bán Giám Sát (Semi-Supervised Learning)

**Mục tiêu:**
- Giả lập kịch bản thiếu nhãn (10-30% labeled)
- So sánh Supervised-only vs Semi-supervised
- Learning curve theo % nhãn
- Phân tích pseudo-label sai

In [ ]:
import sys
sys.path.insert(0, '..')
import os
from pathlib import Path
if Path.cwd().name == 'notebooks':
    os.chdir('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

from src.data.loader import load_params
params = load_params()
seed = params['seed']

## 4b.1 Chuẩn bị dữ liệu

In [ ]:
df_clean = pd.read_csv(params['paths']['processed_data'])

from src.features.builder import select_features_for_modeling
X, y = select_features_for_modeling(df_clean)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=params['split']['test_size'],
    stratify=y, random_state=seed
)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 4b.2 Learning Curve theo % nhãn

In [ ]:
from src.models.semi_supervised import learning_curve_by_label_ratio

lc_results = learning_curve_by_label_ratio(
    X_train_scaled, y_train.values, X_test_scaled, y_test.values,
    label_ratios=params['semi_supervised']['label_ratios'],
    random_state=seed
)

In [ ]:
lc_results.round(4)

In [ ]:
# Visualization: Learning Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F1 Score
axes[0].plot(lc_results['label_ratio']*100, lc_results['supervised_f1'], 'bo-', label='Supervised-only', linewidth=2)
axes[0].plot(lc_results['label_ratio']*100, lc_results['self_training_f1'], 'rs-', label='Self-Training', linewidth=2)
axes[0].plot(lc_results['label_ratio']*100, lc_results['label_spreading_f1'], 'g^-', label='Label Spreading', linewidth=2)
axes[0].set_xlabel('% Labeled Data')
axes[0].set_ylabel('F1 Score')
axes[0].set_title('F1 Score vs Label Ratio')
axes[0].legend()
axes[0].grid(True)

# PR-AUC
axes[1].plot(lc_results['label_ratio']*100, lc_results['supervised_pr_auc'], 'bo-', label='Supervised-only', linewidth=2)
axes[1].plot(lc_results['label_ratio']*100, lc_results['self_training_pr_auc'], 'rs-', label='Self-Training', linewidth=2)
axes[1].plot(lc_results['label_ratio']*100, lc_results['label_spreading_pr_auc'], 'g^-', label='Label Spreading', linewidth=2)
axes[1].set_xlabel('% Labeled Data')
axes[1].set_ylabel('PR-AUC')
axes[1].set_title('PR-AUC vs Label Ratio')
axes[1].legend()
axes[1].grid(True)

plt.suptitle('Semi-Supervised Learning Curve', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/figures/semi_supervised_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 4b.3 Phân tích Pseudo-Labels

In [ ]:
from src.models.semi_supervised import (
    create_partial_labels, run_self_training, analyze_pseudo_labels
)

# Dùng 20% nhãn
y_partial, labeled_mask = create_partial_labels(y_train.values, label_ratio=0.20, random_state=seed)

# Self-training
st_result = run_self_training(X_train_scaled, y_partial, X_test_scaled, y_test.values, seed)

# Phân tích pseudo-labels
analysis = analyze_pseudo_labels(y_train.values, st_result['pseudo_labels'], labeled_mask)

In [ ]:
# Lưu kết quả
from src.evaluation.report import save_results_table
save_results_table(lc_results, 'semi_supervised_learning_curve')

## 4b.4 Nhận xét

- **Self-Training** cho kết quả gần sát Supervised khi có ≥ 15-20% nhãn
- **Label Spreading** kém hơn trong dataset này
- Pseudo-label accuracy tương đối tốt, nhưng có rủi ro FN
- Khi % nhãn thấp (5-10%), semi-supervised giúp cải thiện đáng kể so với chỉ dùng supervised trên ít data